In [1]:
#| tags: [parameters]

# Default values for standalone execution
# When running via Snakemake, these will be overridden by the -P flags
truvari_input_files = []
happy_input_files = []


In [2]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import yaml
from itables import show

# Configure Plotly for notebook and HTML rendering
import plotly.io as pio
pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"
pio_config = {'responsive': True, 'displayModeBar': True, 'displaylogo': False}

# Configure itables for better HTML display
import itables.options as opt
opt.lengthMenu = [10, 25, 50, 100]
opt.maxBytes = 0  # No limit on data size

::: {.panel-tabset}

## Truvari SV Benchmarking

This notebook compares precision, recall, and F1-score metrics across multiple samples and callers from Truvari benchmarking results.


In [3]:
# Load config - either from Snakemake or standalone
try:
    # Running via Snakemake - use snakemake.config
    config = snakemake.config
    print("Running in Snakemake mode - using snakemake.config")
except NameError:
    # Running standalone - load config file manually
    print("Running in standalone mode - searching for config file")
    possible_configs = [
        Path('config/config_nallo_hifi_wgs.yaml'),
        Path('config/config.yaml'),
        Path('../config/config_nallo_hifi_wgs.yaml'),
        Path('../config/config.yaml')
    ]
    
    config_file = None
    for cfg in possible_configs:
        if cfg.exists():
            config_file = cfg
            break
    
    if config_file is None:
        raise FileNotFoundError("Could not find config file. Tried: " + ", ".join(str(p) for p in possible_configs))
    
    print(f"Using config file: {config_file}")
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)

# Get SV callers and suffixes from config
sv_callers = config.get('sv_callers', [])
truvari_vcf_suffixes = config.get('truvari_vcf_suffix', [])

# Ensure suffixes is a list
if isinstance(truvari_vcf_suffixes, str):
    truvari_vcf_suffixes = [truvari_vcf_suffixes]

print(f"SV Callers: {sv_callers}")
print(f"VCF Suffixes: {truvari_vcf_suffixes}")

Running in standalone mode - searching for config file
Using config file: config/config_nallo_hifi_wgs.yaml
SV Callers: ['sniffles', 'sawfish']
VCF Suffixes: ['_sniffles_svs.vcf.gz', '_sawfish_svs.vcf.gz']


In [ ]:
#| tags: [parameters]

# Default values for standalone execution
# When running via Snakemake, these will be overridden by the -P flags 
# or by snakemake.input
truvari_input_files = []
happy_input_files = []

# Get Truvari stats files - either from Snakemake inputs or auto-discovery
import json

file_paths = {}
sample_to_caller = {}

# Priority order: 
# 1. Check if snakemake.input exists
# 2. Check if truvari_input_files was passed as a parameter
# 3. Auto-discover
try:
    # Option 1: Running via Snakemake directly (snakemake-jupyter plugin)
    print("Checking for snakemake.input.truvari_stats...")
    stats_files = snakemake.input.truvari_stats if isinstance(snakemake.input.truvari_stats, list) else [snakemake.input.truvari_stats]
    mode = "snakemake"
except (NameError, AttributeError):
    # Option 2: Running via Quarto with parameters passed via -P
    if isinstance(truvari_input_files, (list, str)) and len(truvari_input_files) > 0:
        print("Using Quarto parameters for Truvari")
        # Quarto might pass a space-separated string if Snakemake expands {input}
        if isinstance(truvari_input_files, str):
            # Split by space, accounting for potential quotes
            import shlex
            stats_files = shlex.split(truvari_input_files)
        else:
            stats_files = truvari_input_files
        mode = "quarto"
    else:
        # Option 3: Standalone auto-discovery
        print("Running in standalone mode - auto-discovering Truvari files")
        mode = "standalone"
        stats_files = []
        
        # Try multiple possible locations for validation_results
        possible_dirs = [
            Path('validation_results'),
            Path('../validation_results'),
            Path('results'),
            Path('../results')
        ]
        
        for base_dir in possible_dirs:
            if base_dir.exists():
                print(f"Scanning {base_dir} for Truvari stats...")
                # Search for the accuracy stats CSV files
                stats_files.extend(list(base_dir.glob("**/ga4gh_with_refine.size_stratified.accuracy.stats.csv")))
        
        if not stats_files:
             print("Warning: No Truvari stats files found in standalone mode!")

# Process the files
for csv_file in stats_files:
    csv_path = Path(csv_file)
    # Extract sample name from path: results/truvari_HM24385-1_sniffles/...
    truvari_dir = csv_path.parent
    full_sample = truvari_dir.name.replace('truvari_', '')  
    
    file_paths[full_sample] = csv_path
    
    # Parse caller from directory name (format: sample_caller or sample-caller)
    caller_id = "unknown"
    for caller in sv_callers:
        if f"_{caller}" in full_sample or f"-{caller}" in full_sample:
            caller_id = caller
            break
    
    sample_to_caller[full_sample] = caller_id
    print(f"Found: {full_sample} -> {csv_path} (caller: {caller_id})")


Checking for snakemake.input.truvari_stats...
Running in standalone mode - auto-discovering Truvari files
Scanning validation_results for Truvari stats...


Scanning results for Truvari stats...
Found: HM24385-2 -> validation_results/truvari_HM24385-2/ga4gh_with_refine.size_stratified.accuracy.stats.csv (caller: unknown)
Found: HM24385-1 -> validation_results/truvari_HM24385-1/ga4gh_with_refine.size_stratified.accuracy.stats.csv (caller: unknown)


In [ ]:
# Read CSV files and add sample identifier and caller
dfs = []

for full_sample, path in file_paths.items():
    if path.exists():
        df = pd.read_csv(path)
        caller = sample_to_caller.get(full_sample, 'unknown')
        
        # Extract biological sample name more robustly by removing _caller or -caller
        bio_sample = full_sample
        if caller != "unknown":
            for sep in ['_', '-']:
                target = f"{sep}{caller}"
                if target in bio_sample:
                    bio_sample = bio_sample.replace(target, "")
                    break
        
        df['Sample'] = full_sample  # Full name (e.g., HM24385-1_sniffles)
        df['BiologicalSample'] = bio_sample  # Just the ID (e.g., HM24385-1)
        df['Caller'] = caller
        dfs.append(df)
        print(f"Loaded {len(df)} rows from {full_sample} (bio: {bio_sample}, caller: {caller})")
    else:
        print(f"Warning: File not found for {full_sample}: {path}")

# Combine all dataframes
combined_df = pd.concat(dfs, ignore_index=True)

print(f"\nTotal rows: {len(combined_df)}")
print(f"Full samples: {combined_df['Sample'].unique()}")
print(f"Biological samples: {combined_df['BiologicalSample'].unique()}")
print(f"Callers: {combined_df['Caller'].unique()}")


Loaded 9 rows from HM24385-2 (bio: HM24385-2, caller: unknown)


Loaded 9 rows from HM24385-1 (bio: HM24385-1, caller: unknown)

Total rows: 18
Full samples: ['HM24385-2' 'HM24385-1']
Biological samples: ['HM24385-2' 'HM24385-1']
Callers: ['unknown']


In [6]:
# Export combined results to Excel (in the publish directory if running via Snakemake)
try:
    # Use the Snakemake output if available, otherwise default to results
    output_file = Path(snakemake.output.excel) if 'snakemake' in globals() else Path('results/combined_truvari_metrics.xlsx')
    output_file.parent.mkdir(parents=True, exist_ok=True)
    combined_df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"Combined metrics exported to: {output_file}")
except Exception as e:
    # Fallback for standalone mode
    output_file = 'combined_truvari_metrics.xlsx'
    combined_df.to_excel(output_file, index=False, engine='openpyxl')
    print(f"Combined metrics exported to: {output_file} (standalone fallback)")

Combined metrics exported to: results/combined_truvari_metrics.xlsx


### Summary Table SV benchmarking HG002 Q100 t2t SV v1.1

In [7]:
# Filter for overall metrics (size-range = All)
overall_df = combined_df[combined_df['size-range'] == 'All'].copy()

overall_summary = overall_df[['BiologicalSample', 'Caller', 'SV-type', 'TP_base', 'TP_comp', 'FP', 'FN', 
                               'Precision', 'Recall', 'F1']].copy()

# Display with interactive table
show(overall_summary, 
     columnDefs=[
         {"className": "dt-center", "targets": "_all"},
         {"width": "10%", "targets": [0, 1, 2]}
     ],
     classes="display compact")

Loading ITables v2.7.0 from the internet... (need help?)


### Visualize Metrics: Overall Comparison (size-range = All)

Compare performance across different SV callers for the same sample.

In [8]:
# Compare callers across all biological samples using faceting
# Get unique callers
unique_callers = overall_df['Caller'].unique()

# Create a combined identifier for sample and caller
overall_df['Sample_Caller'] = overall_df['BiologicalSample'] + ' (' + overall_df['Caller'] + ')'

# Reshape data for plotly express faceting
metrics_df = overall_df.melt(
    id_vars=['BiologicalSample', 'Caller', 'Sample_Caller', 'SV-type'],
    value_vars=['Precision', 'Recall', 'F1'],
    var_name='Metric',
    value_name='Score'
)

# Create faceted bar plot
fig = px.bar(
    metrics_df,
    x='SV-type',
    y='Score',
    color='Sample_Caller',
    facet_row='Caller',
    facet_col='Metric',
    barmode='group',
    color_discrete_sequence=px.colors.qualitative.Plotly + px.colors.qualitative.Set2,
    labels={'SV-type': 'SV Type', 'Score': 'Score', 'Sample_Caller': 'Sample (Caller)'},
    height=400 * len(unique_callers)
)

# Update y-axes to have consistent range
fig.update_yaxes(range=[0, 1.0])

# Update layout
fig.update_layout(
    showlegend=True,
    autosize=True,
    legend=dict(
        title=dict(text='Sample (Caller)'),
        orientation='v'
    )
)

fig.show(config=pio_config)


In [9]:
# Create a direct caller comparison for each biological sample
bio_samples = overall_df['BiologicalSample'].unique()

plots_created = False
for bio_sample in bio_samples:
    bio_sample_data = overall_df[overall_df['BiologicalSample'] == bio_sample]
    callers_for_sample = bio_sample_data['Caller'].unique()
    
    if len(callers_for_sample) > 1:
        plots_created = True
        # Reshape data for plotly express
        metrics_df = bio_sample_data.melt(
            id_vars=['Caller', 'SV-type'],
            value_vars=['Precision', 'Recall', 'F1'],
            var_name='Metric',
            value_name='Score'
        )
        
        # Create faceted bar plot
        fig = px.bar(
            metrics_df,
            x='SV-type',
            y='Score',
            color='Caller',
            facet_col='Metric',
            barmode='group',
            color_discrete_sequence=px.colors.qualitative.Plotly,
            labels={'SV-type': 'SV Type', 'Score': 'Score'},
            title=f'Caller Comparison for {bio_sample}',
            height=500
        )
        
        # Update y-axes to have consistent range
        fig.update_yaxes(range=[0, 1.0])
        
        fig.update_layout(
            showlegend=True,
            autosize=True
        )
        
        fig.show(config=pio_config)

if not plots_created:
    print("No plots generated: Each biological sample has only one caller.")
    print(f"Biological samples: {bio_samples}")
    print(f"Full samples with callers: {overall_df['Sample'].unique()}")


No plots generated: Each biological sample has only one caller.
Biological samples: ['HM24385-2' 'HM24385-1']
Full samples with callers: ['HM24385-2' 'HM24385-1']


### Precision-Recall Scatter Plot with F1-Score Contours

Visualize the precision-recall tradeoff for each sample and caller, with F1-score contour lines showing regions of equivalent performance.

In [10]:
import numpy as np

# Filter for overall metrics (size-range = All) and only Indel SV-type
pr_df = overall_df[overall_df['SV-type'] == 'Indel'].copy()

# Create combined identifier if not already present
if 'Sample_Caller' not in pr_df.columns:
    pr_df['Sample_Caller'] = pr_df['BiologicalSample'] + ' (' + pr_df['Caller'] + ')'

# Create figure
fig = go.Figure()

# Generate F1-score contour lines
# F1 = 2 * (Precision * Recall) / (Precision + Recall)
f1_levels = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

for f1 in f1_levels:
    # For a given F1 and recall, solve for precision: P = F1*R / (2*R - F1)
    
    # Calculate the minimum recall where this F1 is achievable (where P would be 1.0)
    r_min = f1 / (2 - f1)
    
    # Create a denser sampling that includes the critical point at P=1.0
    # Sample from r_min to 1.0 with more points
    recall_range = np.linspace(r_min, 1.0, 300)
    
    precision_for_f1 = []
    recall_for_f1 = []
    
    for r in recall_range:
        if 2*r > f1:  # Valid only when denominator is positive
            p = (f1 * r) / (2*r - f1)
            if 0 <= p <= 1.0:  # Keep only valid precision values
                precision_for_f1.append(p)
                recall_for_f1.append(r)
    
    # Explicitly add the point at P=1.0, R=r_min if not already included
    if len(precision_for_f1) > 0:
        # Check if we're close to P=1.0 at the start
        if precision_for_f1[0] < 0.999:
            recall_for_f1.insert(0, r_min)
            precision_for_f1.insert(0, 1.0)
    
    # Add contour line
    fig.add_trace(go.Scatter(
        x=recall_for_f1,
        y=precision_for_f1,
        mode='lines',
        name=f'F1={f1:.2f}',
        line=dict(color='lightgray', width=1, dash='dash'),
        showlegend=False,
        hovertemplate=f'F1-score: {f1:.2f}<extra></extra>'
    ))
    
    # Add text annotation for this contour
    # Place label at a point roughly 2/3 along the contour for good visibility
    if len(recall_for_f1) > 0:
        label_idx = len(recall_for_f1) * 2 // 3
        fig.add_annotation(
            x=recall_for_f1[label_idx],
            y=precision_for_f1[label_idx],
            text=f'F1={f1:.2f}',
            showarrow=False,
            font=dict(size=9, color='gray'),
            bgcolor='rgba(255,255,255,0.8)',
            borderpad=2
        )

# Get unique values for colors and symbols
unique_callers = pr_df['Caller'].unique()
unique_bio_samples = pr_df['BiologicalSample'].unique()

# Define symbol mapping for biological samples
symbol_map = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down', 'star']

# Define color palette for callers
caller_colors = px.colors.qualitative.Plotly

# Add scatter points for each combination of sample and caller
for bio_sample in unique_bio_samples:
    bio_data = pr_df[pr_df['BiologicalSample'] == bio_sample]
    
    if len(bio_data) == 0:
        continue
    
    # Get symbol for this biological sample
    sample_idx = list(unique_bio_samples).index(bio_sample)
    symbol = symbol_map[sample_idx % len(symbol_map)]
    
    for caller in unique_callers:
        caller_data = bio_data[bio_data['Caller'] == caller]
        
        if len(caller_data) == 0:
            continue
        
        # Get color for this caller
        caller_idx = list(unique_callers).index(caller)
        color = caller_colors[caller_idx % len(caller_colors)]
        
        # Add scatter trace
        fig.add_trace(go.Scatter(
            x=caller_data['Recall'],
            y=caller_data['Precision'],
            mode='markers',
            name=f'{bio_sample} ({caller})',
            marker=dict(
                size=14,
                symbol=symbol,
                color=color,
                line=dict(width=1, color='white')
            ),
            text=[f"Sample: {s}<br>Caller: {c}<br>Precision: {p:.4f}<br>Recall: {r:.4f}<br>F1: {f:.4f}" 
                  for s, c, p, r, f in zip(
                      caller_data['BiologicalSample'],
                      caller_data['Caller'],
                      caller_data['Precision'],
                      caller_data['Recall'],
                      caller_data['F1']
                  )],
            hovertemplate='%{text}<extra></extra>'
        ))

# Update layout
fig.update_layout(
    title='Precision-Recall Trade-off with F1-Score Contours (Indels)',
    xaxis_title='Recall',
    yaxis_title='Precision',
    xaxis=dict(range=[0, 1.0], constrain='domain'),
    yaxis=dict(range=[0, 1.0], scaleanchor='x', scaleratio=1),
    height=700,
    autosize=True,
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02,
        font=dict(size=9)
    ),
    hovermode='closest'
)

fig.show(config=pio_config)

###  Visualize Metrics: Size-Stratified Comparison

In [11]:
# Filter for stratified metrics (exclude All)
stratified_df = combined_df[combined_df['size-range'] != 'All'].copy()

# Convert metric columns to numeric
for col in ['Precision', 'Recall', 'F1']:
    stratified_df[col] = pd.to_numeric(stratified_df[col], errors='coerce')

# Filter for DEL and INS only (exclude Indel totals if present)
stratified_df = stratified_df[stratified_df['SV-type'].isin(['DEL', 'INS'])].copy()

if len(stratified_df) > 0:
    # Create a combined identifier for sample and caller
    stratified_df['Sample_Caller'] = stratified_df['BiologicalSample'] + ' (' + stratified_df['Caller'] + ')'
    
    # Reshape data for plotly express faceting
    metrics_df = stratified_df.melt(
        id_vars=['BiologicalSample', 'Caller', 'Sample_Caller', 'SV-type', 'size-range'],
        value_vars=['Precision', 'Recall', 'F1'],
        var_name='Metric',
        value_name='Score'
    )
    
    # Create faceted bar plot
    unique_callers = metrics_df['Caller'].unique()
    unique_samples = metrics_df['BiologicalSample'].unique()
    
    # Determine coloring strategy based on number of samples and callers
    if len(unique_callers) > 1 or len(unique_samples) > 1:
        color_var = 'Sample_Caller'
        # Use a larger color palette for combined identifiers
        color_seq = px.colors.qualitative.Plotly + px.colors.qualitative.Set2 + px.colors.qualitative.Pastel
    else:
        color_var = 'BiologicalSample'
        color_seq = px.colors.qualitative.Set2
    
    fig = px.bar(
        metrics_df,
        x='size-range',
        y='Score',
        color=color_var,
        facet_row='SV-type',
        facet_col='Metric',
        barmode='group',
        color_discrete_sequence=color_seq,
        labels={'size-range': 'Size Range', 'Score': 'Score', 'Sample_Caller': 'Sample (Caller)'},
        height=900,
        category_orders={'SV-type': ['DEL', 'INS']}
    )
    
    # Update y-axes to have consistent range
    fig.update_yaxes(range=[0, 1.0])
    
    # Update layout
    fig.update_layout(
        showlegend=True,
        autosize=True,
        legend=dict(
            title=dict(text='Sample (Caller)'),
            orientation='v'
        )
    )
    
    fig.show(config=pio_config)
else:
    print("No stratified data available for plotting")


### Heatmap Comparison: F1-Scores Across All Conditions

### Size-Stratified Comparison: Faceted by Caller

Compare size-stratified performance across callers.

In [12]:
# Create size-stratified plots faceted by caller
# First ensure we have the combined identifier in stratified_df if not already present
if 'Sample_Caller' not in stratified_df.columns:
    stratified_df['Sample_Caller'] = stratified_df['BiologicalSample'] + ' (' + stratified_df['Caller'] + ')'

unique_callers = stratified_df['Caller'].unique()

if len(unique_callers) > 1:
    for sv_type in ['DEL', 'INS']:
        sv_df = stratified_df[stratified_df['SV-type'] == sv_type].copy()
        
        if len(sv_df) == 0:
            continue
        
        # Reshape data for plotly express faceting
        metrics_df = sv_df.melt(
            id_vars=['BiologicalSample', 'Caller', 'Sample_Caller', 'size-range'],
            value_vars=['Precision', 'Recall', 'F1'],
            var_name='Metric',
            value_name='Score'
        )
        
        # Create faceted bar plot
        fig = px.bar(
            metrics_df,
            x='size-range',
            y='Score',
            color='Sample_Caller',
            facet_row='Caller',
            facet_col='Metric',
            barmode='group',
            color_discrete_sequence=px.colors.qualitative.Plotly + px.colors.qualitative.Set2,
            labels={'size-range': 'Size Range', 'Score': 'Score', 'Sample_Caller': 'Sample (Caller)'},
            title=f'Size-Stratified Metrics for {sv_type} (by Caller)',
            height=450 * len(unique_callers)
        )
        
        # Update y-axes to have consistent range
        fig.update_yaxes(range=[0, 1.0])
        
        # Update layout
        fig.update_layout(
            showlegend=True,
            autosize=True,
            legend=dict(
                title=dict(text='Sample (Caller)'),
                orientation='v'
            )
        )
        
        fig.show(config=pio_config)
else:
    print("Only one caller found - skipping caller-faceted size-stratified plots")


Only one caller found - skipping caller-faceted size-stratified plots


### Export Combined Results to Excel


In [13]:
# Create separate heatmaps for each caller
unique_callers_all = combined_df['Caller'].unique()

for caller in unique_callers_all:
    caller_df = combined_df[combined_df['Caller'] == caller].copy()
    
    # Ensure 'Condition' exists in this slice (re-create if missing)
    caller_df['Condition'] = caller_df['SV-type'] + ' ' + caller_df['size-range']
    
    # Convert F1 to numeric to ensure pivot works correctly
    caller_df['F1'] = pd.to_numeric(caller_df['F1'], errors='coerce')
    
    # Create pivot table for heatmap
    pivot_data = caller_df.pivot_table(
        values='F1', 
        index='Condition', 
        columns='BiologicalSample'
    )
    
    # Create heatmap using Plotly
    fig = go.Figure(data=go.Heatmap(
        z=pivot_data.values,
        x=pivot_data.columns,
        y=pivot_data.index,
        colorscale='RdYlGn',
        zmin=0,
        zmax=1,
        text=pivot_data.values,
        texttemplate='%{text:.4f}',
        textfont={"size": 10},
        colorbar=dict(title='F1-Score')
    ))
    
    fig.update_layout(
        title=f'F1-Score Heatmap: {caller.upper()} Caller',
        title_x=0.5,
        xaxis_title='Biological Sample',
        yaxis_title='SV Type and Size Range',
        height=600,
        autosize=True
    )
    
    fig.show(config=pio_config)

## Happy SNV/Indel Benchmarking


SNV and Indel benchmarking with hap.py

In [ ]:
# Ingest Happy extended.csv files
# 1. Native Snakemake injection (snakemake.input.happy_stats)
# 2. Quarto Parameters via -P (happy_input_files)
# 3. Auto-discovery fallback
happy_file_paths = {}

try:
    # Option 1: Running via Snakemake directly
    happy_stats_files = snakemake.input.happy_stats if isinstance(snakemake.input.happy_stats, list) else [snakemake.input.happy_stats]
    print("Using Snakemake inputs for Happy")
except (NameError, AttributeError):
    # Option 2: Running via Quarto with parameters passed via -P
    if 'happy_input_files' in globals() and happy_input_files:
        print("Using Quarto parameters for Happy")
        if isinstance(happy_input_files, str):
            import shlex
            happy_stats_files = shlex.split(happy_input_files)
        else:
            happy_stats_files = happy_input_files
    else:
        # Option 3: Standalone auto-discovery
        print("Running in standalone mode - auto-discovering Happy files")
        happy_stats_files = []
        possible_dirs = [Path('validation_results'), Path('../validation_results'), Path('results'), Path('../results')]
        for vdir in possible_dirs:
            if vdir.exists():
                happy_stats_files.extend(list(vdir.glob("**/happy_*/*.out.extended.csv")))

# Process and load the Happy files
happy_dfs = []
for csv_file in happy_stats_files:
    csv_path = Path(csv_file)
    if not csv_path.exists():
        continue
        
    full_sample = csv_path.parent.name.replace('happy_', '')
    
    # Strip caller suffix for clean SNV/Indel labels
    clean_sample = full_sample
    for caller in sv_callers:
        if full_sample.endswith(f"-{caller}"):
            clean_sample = full_sample[:-len(f"-{caller}")]
        elif full_sample.endswith(f"_{caller}"):
            clean_sample = full_sample[:-len(f"_{caller}")]
    
    df = pd.read_csv(csv_path)
    df['Sample'] = clean_sample
    happy_dfs.append(df)
    print(f"Found Happy Sample: {clean_sample}")

if happy_dfs:
    happy_combined_df = pd.concat(happy_dfs, ignore_index=True)
    happy_pass_df = happy_combined_df[happy_combined_df['Filter'] == 'PASS'].copy()
    print(f"Loaded {len(happy_pass_df)} PASS records.")
else:
    print("No Happy results found.")

### Happy F1-score Heatmaps of GIAB stratifications
These heatmaps show metrics across all stratification subsets (e.g., GC content, Repetitive regions) for each type of variant (SNV, INDEL).


In [16]:
# Create heatmaps for SNV and INDEL separately for better readability (F1-Score only)
for v_type in sorted(happy_pass_df['Type'].unique()):
    v_data = happy_pass_df[happy_pass_df['Type'] == v_type].copy()
    
    # Happy extended.csv has Subtype (e.g., '*' for overall, or specific lengths/types)
    # To avoid duplicates in pivot, we filter for Subtype == '*' (the summary for the Subset)
    v_data = v_data[v_data['Subtype'] == '*'].copy()
    
    # Exclude HG002-specific stratification subsets and "notin" records
    v_data = v_data[
        (~v_data['Subset'].str.contains('HG00', case=False, na=False)) & 
        (~v_data['Subset'].str.contains('notin', case=False, na=False))
    ].copy()

    # Create pivot table for the F1-Score metric using the simplified 'Sample' labels
    pivot_vals = v_data.pivot(index='Subset', columns='Sample', values='METRIC.F1_Score')
    
    # Remove subsets (rows) that have NA values for all samples to clean up the plot
    pivot_vals = pivot_vals.dropna(how='all')

    # Sort subsets by mean F1-Score across all samples (lowest at the top)
    row_means = pivot_vals.mean(axis=1)
    pivot_vals = pivot_vals.reindex(row_means.sort_values(ascending=True).index)
    
    # Plotly heatmap with custom color scale midpoint at 0.5
    fig_hm = px.imshow(
        pivot_vals, 
        labels=dict(x="Sample", y="Stratification Subset", color="F1-Score"),
        title=f"Happy {v_type} F1-Score across Subsets (Subtype=*) - Sorted by performance",
        color_continuous_scale="RdYlGn", 
        color_continuous_midpoint=0.5,
        zmin=0.9, zmax=1, 
        aspect="auto"
    )
    
    # Automatically adjust height based on number of rows to keep labels readable
    calculated_height = max(400, len(pivot_vals) * 20)
    fig_hm.update_layout(height=calculated_height) 
    fig_hm.show(config=pio_config)


:::
